In [1]:
from __future__ import annotations

import math
import os
from pathlib import Path
from typing import Dict, Tuple, Optional, List

import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import xy, rowcol
from pyproj import Transformer

from geotessera import GeoTessera
import geopandas as gpd
import time
from scipy.ndimage import distance_transform_edt
from skimage.measure import block_reduce

import os
os.environ['AWS_PROFILE'] = 'ctrees'
os.environ['GDAL_DISABLE_READDIR_ON_OPEN'] = 'YES'
os.environ['VSI_CURL_ALLOW_CREDENTIALS'] = 'YES'

import boto3
s3 = boto3.client('s3')


In [2]:
# Get training tiles
training_tiles_path = 's3://ctrees-development/activity/tessera/planet_nicfi_logging_tessera_pantropical_v1_testing.gpkg'
training_tiles = gpd.read_file(training_tiles_path)
tiles_id = training_tiles['name'].tolist()
print(f"There are {len(tiles_id)} training tiles.")

There are 10 training tiles.


In [3]:
# open reinforced samples
# get all s3 files in training_parquet_prefix
s3 = boto3.client('s3')
bucket_name = 'ctrees-development'
prefix = 'activity/tessera/logging_pantropical_v1/reinforced_samples/'
response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.parquet')]
print(files)


['activity/tessera/logging_pantropical_v1/reinforced_samples/-54.35_-11.65_samples.parquet', 'activity/tessera/logging_pantropical_v1/reinforced_samples/-54.35_-11.75_samples.parquet', 'activity/tessera/logging_pantropical_v1/reinforced_samples/-54.35_-11.85_samples.parquet', 'activity/tessera/logging_pantropical_v1/reinforced_samples/-54.45_-11.65_samples.parquet', 'activity/tessera/logging_pantropical_v1/reinforced_samples/-54.45_-11.75_samples.parquet', 'activity/tessera/logging_pantropical_v1/reinforced_samples/-54.45_-11.85_samples.parquet', 'activity/tessera/logging_pantropical_v1/reinforced_samples/-55.85_-3.55_samples.parquet', 'activity/tessera/logging_pantropical_v1/reinforced_samples/-55.85_-3.65_samples.parquet', 'activity/tessera/logging_pantropical_v1/reinforced_samples/-55.95_-3.55_samples.parquet', 'activity/tessera/logging_pantropical_v1/reinforced_samples/-55.95_-3.65_samples.parquet', 'activity/tessera/logging_pantropical_v1/reinforced_samples/-56.05_-3.55_samples.pa

In [8]:
gt = GeoTessera()
def extract_samples_per_tessera_tile(sample_path: str) -> pd.DataFrame:
    samples = pd.read_parquet(sample_path, storage_options={'profile': 'ctrees'})
    tessera_id = samples['tessera_id'].unique()[0]
    lon_id = float(tessera_id.split('_')[0])
    lat_id = float(tessera_id.split('_')[1])

    # fetch tesssera tile
    year = 2024
    embedding, crs, transform = gt.fetch_embedding(lon=lon_id, lat=lat_id, year=year)
    # Extract all embedding vectors at once with numpy fancy indexing,
    # then concat as a single DataFrame to avoid per-column fragmentation.
    rows = samples['row'].to_numpy()
    cols = samples['col'].to_numpy()
    emb_matrix = embedding[rows, cols, :]  # shape (n_samples, n_dims)

    emb_cols = [f'emb_{i:03d}' for i in range(emb_matrix.shape[1])]
    emb_df = pd.DataFrame(emb_matrix, columns=emb_cols, index=samples.index)

    return pd.concat([samples, emb_df], axis=1)


In [5]:
ff = files[0]
ff

'activity/tessera/logging_pantropical_v1/reinforced_samples/-54.35_-11.65_samples.parquet'

In [ ]:

for f in files:
    t0 = time.time()
    
    output_bucket = 'ctrees-development'
    output_prefix = 'activity/tessera/logging_pantropical_v1/extracted_embeddings_reinforced_samples'
    file_name = f.split('/')[-1]
    output_path = f's3://{output_bucket}/{output_prefix}/{file_name}'
    print(f"Processing file: {f}")

    sample_path= f's3://{bucket_name}/{f}'
    samples = extract_samples_per_tessera_tile(sample_path)

    # export to s3 as parquet:
    samples.to_parquet(output_path,  storage_options={'profile': 'ctrees'},index=False)
    print(f"Exported processed dataframe to {output_path}")

    # remove the downloaded embedding folder path:
    working_dir = os.getcwd()
    embedding_folder = os.path.join(working_dir, "global_0.1_degree_representation")
    landmark_folder = os.path.join(working_dir, "global_0.1_degree_tiff_all")
    if os.path.exists(embedding_folder):
        import shutil
        shutil.rmtree(embedding_folder)
        print(f"Removed embedding folder: {embedding_folder}")
    else:
        print(f"Embedding folder not found, skipping removal: {embedding_folder}")


    if os.path.exists(landmark_folder):
        import shutil
        shutil.rmtree(landmark_folder)
        print(f"Removed landmark folder: {landmark_folder}")
    else:
        print(f"Landmark folder not found, skipping removal: {landmark_folder}")
    print(f"Finished processing file: {f} in {time.time() - t0:.2f} seconds")
    del samples
